# 🌲 XGBoost Pure-Rust Tree Inference with SnoozeML

This notebook demonstrates:
1. **Train** an XGBoost gradient boosted tree on the standard Breast Cancer dataset.
2. **Export** directly to JSON via `snoozeml.converters.export_xgboost`.
3. **Upload / Deploy** via **HTTP upload** directly to SnoozeML's native pure-Rust `hessboost` tree runtime (zero C++ dependencies, no volume mounting or restart needed).
4. **Cold Start vs. Warm Execution**: Compare cold model loading vs. ultra-fast in-memory tree evaluation.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import xgboost as xgb

from snoozeml import SnoozeClient
from snoozeml.converters import export_xgboost

## 1. Load Data & Train XGBoost

In [2]:
# Load Breast Cancer dataset (30 features, binary classification)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit an XGBoost classifier
model = xgb.XGBClassifier(n_estimators=25, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
print(f"Local train accuracy: {model.score(X_train, y_train):.4f}")

Local train accuracy: 0.9912


## 2. Export & Upload Directly to SnoozeML via HTTP

Export to JSON format and upload directly to SnoozeML's REST API without requiring volume mounting or container restarts.

In [3]:
# Export tree model locally to native JSON format
model_path = Path("./cancer_xgboost.xgb.json")
export_xgboost(model, output_path=model_path)
print(f"Exported tree model to {model_path} ({model_path.stat().st_size} bytes)")

# Connect to SnoozeML daemon
# If authentication is enabled, pass your API key or JWT token:
# client = SnoozeClient(endpoint="http://localhost:8080", api_key="snooze_admin_secret_key")
client = SnoozeClient(endpoint="http://localhost:8080")

# Upload model directly via HTTP with custom runtime configuration
model_info = client.upload_model(
    file=model_path,
    model_id="cancer_xgboost",
    description="XGBoost Classifier on Breast Cancer Dataset (Pure-Rust trees)",
    pinned=False,               # Keep unpinned to allow scale-to-zero memory reclamation
    max_batch_size=16,
    max_batch_delay_ms=10,
    idle_timeout_secs=300,
)
print(f"Uploaded model: {model_info.id} | Initial State: {model_info.state.value} | Pinned: {model_info.pinned}")

# Note: Runtime model refresh can also be triggered at any time:
# client.refresh_models()

Exported tree model to cancer_xgboost.xgb.json (28243 bytes)
Uploaded model: cancer_xgboost | Initial State: idle | Pinned: False


## 3. Compare Cold Start vs. Warm In-Memory Inference

Demonstrating SnoozeML's dynamic loading on cold start and blazing fast warm in-memory inference.

In [4]:
# Ensure model starts in idle state (scale-to-zero)
client.models.evict("cancer_xgboost")
print(f"State before 1st request: {client.models.get('cancer_xgboost').state.value}")

# --- 1st Request: Cold Start (JSON tree parsing + model allocation into memory) ---
resp_cold = client.infer(model_id="cancer_xgboost", inputs={"features": X_test})
print(f"❄️  Cold Start Latency: {resp_cold.duration_ms:.2f} ms (tree weights loaded on-demand)")
print(f"State after 1st request:  {client.models.get('cancer_xgboost').state.value}")

# --- 2nd Request: Warm Execution (pure-Rust hessboost evaluation in memory) ---
resp_warm = client.infer(model_id="cancer_xgboost", inputs={"features": X_test})
print(f"🔥 Warm Latency:        {resp_warm.duration_ms:.2f} ms (in-memory tree evaluation)")

speedup = resp_cold.duration_ms / max(resp_warm.duration_ms, 0.01)
print(f"⚡ Speedup:             {speedup:.1f}x faster warm execution")

# Evaluate predictions
probs = resp_warm.get_numpy().flatten()
preds = (probs >= 0.5).astype(int)

print(f"Test ROC-AUC:      {roc_auc_score(y_test, probs):.4f}")
print(f"Test Accuracy:     {accuracy_score(y_test, preds):.4f}")
print(f"Predicted labels:  {preds[:5]}")
print(f"Ground truth:      {y_test[:5]}")

State before 1st request: idle
❄️  Cold Start Latency: 6.03 ms (tree weights loaded on-demand)
State after 1st request:  ready
🔥 Warm Latency:        0.21 ms (in-memory tree evaluation)
⚡ Speedup:             28.4x faster warm execution
Test ROC-AUC:      0.9934
Test Accuracy:     0.9561
Predicted labels:  [1 0 0 1 1]
Ground truth:      [1 0 0 1 1]
